In [11]:
from twelvedata import TDClient
import pandas as pd
import numpy as np

In [12]:
# Initialize client - apikey parameter is requiered
td = TDClient(apikey="a5a6a89a52b240b7ae34e9926ac2daef")

In [6]:

# Construct the necessary time series
ts = td.time_series(
    symbol=['EUR/USD', 'GBP/USD', 'USD/CHF', 'USD/JPY', 'USD/CAD', 'AUD/USD', 'NZD/USD'],
    interval="30min",
    outputsize=15
)

# Returns pandas.DataFrame
df = ts.as_pandas()
df

open     high      low    close
EUR/USD 2026-01-21 07:30:00  1.17193  1.17199  1.17170  1.17174
        2026-01-21 07:00:00  1.17131  1.17202  1.17109  1.17192
        2026-01-21 06:30:00  1.17229  1.17232  1.17133  1.17133
        2026-01-21 06:00:00  1.17277  1.17280  1.17212  1.17229
        2026-01-21 05:30:00  1.17319  1.17334  1.17267  1.17279
...                              ...      ...      ...      ...
NZD/USD 2026-01-21 02:30:00  0.58434  0.58499  0.58432  0.58482
        2026-01-21 02:00:00  0.58356  0.58442  0.58342  0.58433
        2026-01-21 01:30:00  0.58335  0.58384  0.58334  0.58356
        2026-01-21 01:00:00  0.58337  0.58365  0.58313  0.58335
        2026-01-21 00:30:00  0.58326  0.58388  0.58326  0.58337

[105 rows x 4 columns]

In [7]:
# Check dataframe structure
print("DataFrame shape:", df.shape)
print("\nDataFrame columns:")
print(df.columns)
print("\nDataFrame index:")
print(df.index)
print("\nFirst few rows:")
df.head()

DataFrame shape: (105, 4)

DataFrame columns:
Index(['open', 'high', 'low', 'close'], dtype='object')

DataFrame index:
MultiIndex([('EUR/USD', '2026-01-21 07:30:00'),
            ('EUR/USD', '2026-01-21 07:00:00'),
            ('EUR/USD', '2026-01-21 06:30:00'),
            ('EUR/USD', '2026-01-21 06:00:00'),
            ('EUR/USD', '2026-01-21 05:30:00'),
            ('EUR/USD', '2026-01-21 05:00:00'),
            ('EUR/USD', '2026-01-21 04:30:00'),
            ('EUR/USD', '2026-01-21 04:00:00'),
            ('EUR/USD', '2026-01-21 03:30:00'),
            ('EUR/USD', '2026-01-21 03:00:00'),
            ...
            ('NZD/USD', '2026-01-21 05:00:00'),
            ('NZD/USD', '2026-01-21 04:30:00'),
            ('NZD/USD', '2026-01-21 04:00:00'),
            ('NZD/USD', '2026-01-21 03:30:00'),
            ('NZD/USD', '2026-01-21 03:00:00'),
            ('NZD/USD', '2026-01-21 02:30:00'),
            ('NZD/USD', '2026-01-21 02:00:00'),
            ('NZD/USD', '2026-01-21 01:30:00'),


open     high      low    close
EUR/USD 2026-01-21 07:30:00  1.17193  1.17199  1.17170  1.17174
        2026-01-21 07:00:00  1.17131  1.17202  1.17109  1.17192
        2026-01-21 06:30:00  1.17229  1.17232  1.17133  1.17133
        2026-01-21 06:00:00  1.17277  1.17280  1.17212  1.17229
        2026-01-21 05:30:00  1.17319  1.17334  1.17267  1.17279

In [10]:
# Method 1: If dataframe has MultiIndex with symbol as index level
# (common with twelvedata API)
if isinstance(df.index, pd.MultiIndex):
    # Group by symbol from index
    grouped = df.groupby(level='symbol')  # or level=0 if symbol is first level
    
    # Iterate through groups
    for symbol, group in grouped:
        print(f"\n{symbol}:")
        print(group.head(2))
    
    # Get specific pair
    eurusd = grouped.get_group('EUR/USD')
    
    # Apply operations to each group
    summary = grouped.agg({
        'close': ['mean', 'std', 'min', 'max'],
        'volume': 'sum'
    })
    print("\nSummary by pair:")
    print(summary)

KeyError: 'Level symbol not found'

In [ ]:
# Method 2: If dataframe has 'symbol' column
if 'symbol' in df.columns:
    grouped = df.groupby('symbol')
    
    # Iterate through groups
    for symbol, group in grouped:
        print(f"\n{symbol}: {len(group)} rows")
    
    # Get specific pair
    eurusd = df[df['symbol'] == 'EUR/USD']
    
    # Apply aggregations
    summary = df.groupby('symbol').agg({
        'close': ['mean', 'std', 'min', 'max'],
        'volume': 'sum',
        'open': 'first',
        'close': 'last'
    })
    print("\nSummary by pair:")
    print(summary)

In [ ]:
# Method 3: If dataframe has 'instrument' column (like in train_model_pipeline)
# This is common when you concatenate multiple pairs
if 'instrument' in df.columns:
    grouped = df.groupby('instrument')
    
    # Get all unique pairs
    pairs = df['instrument'].unique()
    print("Available pairs:", pairs)
    
    # Iterate through groups
    for pair, group in grouped:
        print(f"\n{pair}: {len(group)} rows")
    
    # Get specific pair
    eurusd = df[df['instrument'] == 'EURUSD']
    
    # Apply operations to each group
    summary = df.groupby('instrument').agg({
        'close': ['mean', 'std'],
        'volume': 'sum',
        'high': 'max',
        'low': 'min'
    })
    print("\nSummary by pair:")
    print(summary)

In [ ]:
# Method 4: Group by multiple columns (base_currency and quote_currency)
if 'base_currency' in df.columns and 'quote_currency' in df.columns:
    grouped = df.groupby(['base_currency', 'quote_currency'])
    
    # Iterate through groups
    for (base, quote), group in grouped:
        print(f"\n{base}/{quote}: {len(group)} rows")
    
    # Get specific pair
    eurusd = df[(df['base_currency'] == 'EUR') & (df['quote_currency'] == 'USD')]
    
    # Summary by currency pair
    summary = df.groupby(['base_currency', 'quote_currency']).agg({
        'close': 'mean',
        'volume': 'sum'
    })
    print("\nSummary by currency pair:")
    print(summary)

In [ ]:
# Method 5: Common operations after grouping
# Example: Calculate statistics for each pair

# If using MultiIndex
if isinstance(df.index, pd.MultiIndex):
    grouped = df.groupby(level='symbol')
elif 'symbol' in df.columns:
    grouped = df.groupby('symbol')
elif 'instrument' in df.columns:
    grouped = df.groupby('instrument')
else:
    print("No symbol/instrument column found. Please check dataframe structure.")
    grouped = None

if grouped is not None:
    # Calculate various statistics per pair
    stats_per_pair = grouped.agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum',
        'close': ['mean', 'std', 'count']
    })
    
    print("Statistics per pair:")
    print(stats_per_pair)
    
    # Calculate percentage change per pair
    pct_change_per_pair = grouped['close'].pct_change()
    
    # Calculate rolling statistics per pair
    rolling_mean_per_pair = grouped['close'].rolling(window=5).mean()
    
    print("\nRolling mean (5 periods) per pair:")
    print(rolling_mean_per_pair.head(10))

In [ ]:
# Method 6: Apply custom function to each group
def calculate_volatility(group):
    """Calculate volatility for a group"""
    returns = group['close'].pct_change()
    volatility = returns.std() * np.sqrt(252)  # Annualized volatility
    return volatility

if isinstance(df.index, pd.MultiIndex):
    volatility_by_pair = df.groupby(level='symbol').apply(calculate_volatility)
elif 'symbol' in df.columns:
    volatility_by_pair = df.groupby('symbol').apply(calculate_volatility)
elif 'instrument' in df.columns:
    volatility_by_pair = df.groupby('instrument').apply(calculate_volatility)
else:
    volatility_by_pair = None

if volatility_by_pair is not None:
    print("Volatility by pair:")
    print(volatility_by_pair)

In [ ]:
# Method 7: Filter and work with specific pairs
# Get list of all pairs first
if isinstance(df.index, pd.MultiIndex):
    pairs = df.index.get_level_values('symbol').unique()
elif 'symbol' in df.columns:
    pairs = df['symbol'].unique()
elif 'instrument' in df.columns:
    pairs = df['instrument'].unique()
else:
    pairs = []

print("Available pairs:", pairs)

# Work with specific pairs
if len(pairs) > 0:
    # Filter for major pairs only
    major_pairs = ['EUR/USD', 'GBP/USD', 'USD/JPY', 'USD/CHF', 'AUD/USD']
    
    if isinstance(df.index, pd.MultiIndex):
        major_df = df[df.index.get_level_values('symbol').isin(major_pairs)]
    elif 'symbol' in df.columns:
        major_df = df[df['symbol'].isin(major_pairs)]
    elif 'instrument' in df.columns:
        # Convert instrument format if needed (EURUSD -> EUR/USD)
        major_instruments = [p.replace('/', '') for p in major_pairs]
        major_df = df[df['instrument'].isin(major_instruments)]
    
    print(f"\nFiltered to major pairs: {len(major_df)} rows")
    print(major_df.head())

## Quick Reference: Grouping by Forex Pairs

### Most Common Patterns:

1. **MultiIndex (twelvedata format):**
   ```python
   grouped = df.groupby(level='symbol')
   # or
   grouped = df.groupby(level=0)  # if symbol is first level
   ```

2. **Symbol column:**
   ```python
   grouped = df.groupby('symbol')
   ```

3. **Instrument column:**
   ```python
   grouped = df.groupby('instrument')
   ```

4. **Multiple columns:**
   ```python
   grouped = df.groupby(['base_currency', 'quote_currency'])
   ```

### Common Operations:

- **Get specific pair:** `df[df['symbol'] == 'EUR/USD']`
- **Iterate groups:** `for pair, group in grouped: ...`
- **Aggregate:** `grouped.agg({'close': 'mean', 'volume': 'sum'})`
- **Apply function:** `grouped.apply(your_function)`
- **Get group:** `grouped.get_group('EUR/USD')`